In [28]:
!pip install pandas selfies tqdm rdkit


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [61]:
from pathlib import Path
import pandas as pd
import selfies
import os
from tqdm import tqdm

INPUT_PATH  = r"E:/Code/Poliforge/Polyforge-AI/Polyforge-AI/notebooks/data/raw/polymers_with_names_predicted.csv"
OUTPUT_PATH = r"E:/Code/Poliforge/Polyforge-AI/Polyforge-AI/notebooks/data/processed/polymers_with_selfies.csv"
df = pd.read_csv(INPUT_PATH)
print(f"Загружено строк: {len(df)}")
print("Колонки:", df.columns.tolist())
df.head(2)

Загружено строк: 11239
Колонки: ['ID', 'Name', 'Polymer_SMILES', 'Monomer_SMILES_1', 'Monomer_SMILES_2', 'Egc', 'Egb', 'Eib', 'CED', 'Ei', 'Eea', 'nc', 'ne', 'Xc', 'Xe', 'epse_6.0', 'epsc', 'epse_3.0', 'epse_1.78', 'epse_15.0', 'epse_4.0', 'epse_5.0', 'epse_2.0', 'epse_9.0', 'epse_7.0', 'epsb', 'TSb', 'TSy', 'YM', 'permCH4', 'permCO2', 'permH2', 'permO2', 'permN2', 'permHe', 'Cp', 'Td', 'Tg', 'Tm', 'rho', 'LOI']


,ID,Name,Polymer_SMILES,Monomer_SMILES_1,Monomer_SMILES_2,Egc,Egb,Eib,CED,Ei,...,permH2,permO2,permN2,permHe,Cp,Td,Tg,Tm,rho,LOI
0,P010001,polyethene,*C*,C=C,NaN,4.330576,4.113436,3.935148,82.704056,5.762087,...,40.347748,11.179819,4.22428,27.200653,1.33078,588.05804,313.36267,430.56586,1.194377,24.51594
1,P010001,poly(diazomethane),*C*,C=[N+]=[N-],NaN,4.330576,4.113436,3.935148,82.704056,5.762087,...,40.347748,11.179819,4.22428,27.200653,1.33078,588.05804,313.36267,430.56586,1.194377,24.51594


In [59]:
import pandas as pd
import selfies as sf
import re
from tqdm import tqdm

# =================================================================
# Улучшенная функция конвертации
# =================================================================
def robust_smiles_to_selfies(smiles):
    if not isinstance(smiles, str) or smiles.strip() == "":
        return None

    # 1. Защита от изменений оригинала
    s = smiles.strip()

    # 2. Лечим нитрогруппу N(=O)=O -> [N+](=O)[O-]
    # Это самая частая ошибка в химических базах (невалидная валентность азота)
    s = s.replace("N(=O)=O", "[N+](=O)[O-]")

    # 3. Обработка точек полимеризации (*)
    # Заменяем на Ксенон [Xe], так как он разрешен в SELFIES и имеет гибкую валентность
    s = s.replace("[*]", "[Xe]").replace("*", "[Xe]")

    # 4. Попытка первой конвертации
    try:
        return sf.encoder(s).replace("[Xe]", "[*]")
    except:
        pass

    # 5. Если не вышло (ошибка в ароматике), пытаемся "Кекулизацию" азота
    # Заменяем ароматический 'n' на обычный 'N', если он вызывает сбой
    try:
        s_fixed = s.replace("n", "N")
        return sf.encoder(s_fixed).replace("[Xe]", "[*]")
    except:
        pass

    # 6. Последний шанс: если ошибка в сложных ветвлениях типа CC(*)C*
    try:
        # Убираем скобки вокруг точек сочленения, если они есть
        s_alt = s.replace("([Xe])", "[Xe]")
        return sf.encoder(s_alt).replace("[Xe]", "[*]")
    except Exception as e:
        # print(f"Ошибка в SMILES: {smiles} | Ошибка: {e}") # Раскомментируйте для дебага
        return None

# =================================================================
# Основной блок выполнения
# =================================================================

input_path = "E:/Code/Poliforge/Polyforge-AI/Polyforge-AI/notebooks/data/raw/polymers_with_names_predicted.csv"
output_path = "E:/Code/Poliforge/Polyforge-AI/Polyforge-AI/notebooks/data/raw/polymers_with_names_predicted_selfies.csv"

print("Чтение файла...")
df = pd.read_csv(input_path)

print("Запуск конвертации (обработка сложных случаев)...")
tqdm.pandas()
df['SELFIES'] = df['Polymer_SMILES'].progress_apply(robust_smiles_to_selfies)

# Проверка того самого проблемного случая
problematic = 'CC(C(OCCn(c1c2cccc1)c3c2cc(/N=N/c4ccc(N(=O)=O)cc4)cc3)=O)([*])C[*]'
fixed_res = robust_smiles_to_selfies(problematic)

print("\n--- Результат для проблемной строки ---")
print(f"SMILES: {problematic}")
print(f"SELFIES: {fixed_res}")
print("---------------------------------------")

# Статистика
total = len(df)
success = df['SELFIES'].notna().sum()
print(f"\nИтог: {success} из {total} успешно ({success/total*100:.2f}%)")

# Сохранение
df.to_csv(output_path, index=False)
print(f"\nФайл сохранен: {output_path}")

Чтение файла...
Запуск конвертации (обработка сложных случаев)...


100%|██████████| 11239/11239 [00:04<00:00, 2652.74it/s]



--- Результат для проблемной строки ---
SMILES: CC(C(OCCn(c1c2cccc1)c3c2cc(/N=N/c4ccc(N(=O)=O)cc4)cc3)=O)([*])C[*]
SELFIES: [C][C][Branch2][Branch1][C][C][Branch2][Ring2][=N][O][C][C][N][Branch1][=Branch2][C][=C][C][=C][C][=C][Ring1][=Branch1][C][=C][Ring1][#Branch1][C][=C][Branch2][Ring1][C][/N][=N][/C][=C][C][=C][Branch1][=Branch1][N+1][=Branch1][C][=O][O-1][C][=C][Ring1][=Branch2][C][=C][Ring1][P][=O][Branch1][C][*][C][*]
---------------------------------------

Итог: 11239 из 11239 успешно (100.00%)

Файл сохранен: E:/Code/Poliforge/Polyforge-AI/Polyforge-AI/notebooks/data/raw/polymers_with_names_predicted_selfies.csv


In [60]:
df.head(2)

,ID,Name,Polymer_SMILES,Monomer_SMILES_1,Monomer_SMILES_2,Egc,Egb,Eib,CED,Ei,...,permO2,permN2,permHe,Cp,Td,Tg,Tm,rho,LOI,SELFIES
0,P010001,polyethene,*C*,C=C,NaN,4.330576,4.113436,3.935148,82.704056,5.762087,...,11.179819,4.22428,27.200653,1.33078,588.05804,313.36267,430.56586,1.194377,24.51594,[*][C][*]
1,P010001,poly(diazomethane),*C*,C=[N+]=[N-],NaN,4.330576,4.113436,3.935148,82.704056,5.762087,...,11.179819,4.22428,27.200653,1.33078,588.05804,313.36267,430.56586,1.194377,24.51594,[*][C][*]
